# Characterizing Proximal Kinematic Fatigue Profiles Using a Standalone IoMT Wrist Sensor

A dual-output CNN–LSTM that simultaneously classifies **which** upper-limb rehabilitation exercise is being performed and assesses **how well** it is performed, from a single wrist-worn IMU — evaluated under strict leave-one-subject-out cross-validation across 15 subjects and 3,001 repetitions.

> **Note on the data.** The recordings come from human participants under informed consent obtained in accordance with the Declaration of Helsinki and institutional ethics approval. They are **not redistributable** and are not included here. The notebook is published for the method; the expected CSV schema is described in the README.

**What makes the evaluation strict, and why it matters:**

1. **Leave-one-subject-out.** Every reported number is measured on a participant whose data never appeared in training. This is genuine subject-independent generalisation, not a held-out session.
2. **Per-fold ground truth.** The DTW barycentre ("kinematic centroid") and the scaler are fitted **inside each training fold only**, then applied to the held-out subject. Nothing about the test participant informs the labels or the normalisation.
3. **Augmentation after the split.** The training fold is isolated first, then augmented 3× — so no augmented clone of a repetition can appear on both sides of the train/validation boundary.

**The honest headline:** on exercise *taxonomy*, a CNN-only model (88.96%) and even a 1-nearest-neighbour DTW classifier (88.56%) match the full hybrid (88.73%). The hybrid earns its keep on *execution quality*, where the LSTM lifts ROC-AUC from 0.8639 to 0.9060. Section 7 states this plainly rather than burying it.

In [1]:
%pip install tensorflow

In [2]:
!pip install tslearn

## 1. Signal processing and repetition segmentation

Raw IMU streams are segmented into individual repetitions before anything else happens. The segmentation is deliberately conservative, because a mis-split repetition corrupts both the label and the feature vector downstream.

**Why a 0.8 Hz low-pass.** The cutoff was lowered from 1.5 Hz to suppress tremor-band content that otherwise produces spurious peaks inside a single repetition. The trade-off is explicit: a lower cutoff is more robust to tremor but will merge genuinely fast repetitions.

**Why the constraints.** Peak detection uses a *dynamic* prominence of `max(signal_std × 0.20, 0.05)` rather than a fixed threshold, so the segmenter adapts to each participant's signal amplitude instead of assuming a common scale. Valleys are separated by at least 0.8 s, and any candidate repetition shorter than 0.5 s is discarded — both guard against intra-repetition fragmentation.

**One property worth noting:** segmentation runs inside a `groupby(['Subject_ID', 'Exercise'])`, so repetition boundaries never straddle two participants or two exercises. The windowing is correct by construction here.

In [3]:
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks
from scipy.interpolate import CubicSpline
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tslearn.barycenters import dtw_barycenter_averaging
from tslearn.metrics import dtw

# ==============================================================================
# PHASE 0 & 1: SIGNAL PROCESSING & AUTO-SEGMENTATION  (REVISED)
# ==============================================================================

def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

def get_segmentation_signal(df, feature_cols):
    accel_cols = [col for col in feature_cols if 'Acceleration' in col]
    if len(accel_cols) == 3:
        return np.linalg.norm(df[accel_cols].values, axis=1)

    variances = df[feature_cols].var()
    return df[variances.idxmax()].values

def segment_repetitions(df, feature_cols, fs=100, cutoff=0.8):
    raw_signal = get_segmentation_signal(df, feature_cols)

    # Aggressively lowered from 1.5 Hz to 0.8 Hz to crush pathological tremors
    smoothed_signal = butter_lowpass_filter(raw_signal, cutoff, fs)

    signal_std = np.std(smoothed_signal)
    dynamic_prominence = max(signal_std * 0.20, 0.05)

    peaks, _ = find_peaks(smoothed_signal, distance=fs, prominence=dynamic_prominence)

    # Increased distance constraint from 0.5s to 0.8s to prevent intra-repetition fragmentation
    valleys, _ = find_peaks(-smoothed_signal, distance=int(fs * 0.8))

    repetitions = []
    for peak in peaks:
        left = valleys[valleys < peak]
        right = valleys[valleys > peak]
        if len(left) > 0 and len(right) > 0:
            start_idx = left[-1]
            end_idx = right[0]

            # Minimum physical duration constraint
            if (end_idx - start_idx) > (fs * 0.5):
                rep_slice = df.iloc[start_idx:end_idx].copy()
                repetitions.append(rep_slice)

    return repetitions



## 2. Temporal normalisation, derived features, and augmentation

**Resampling to 150 time steps.** Repetitions vary in duration both within and across participants. Linear interpolation to a fixed 150-step length makes them tensor-compatible, at the cost of discarding absolute duration as a feature — a deliberate choice, since duration is partly a property of the person rather than of the movement quality.

**Derived channels (9 → 12).** Acceleration magnitude, gyroscope magnitude and jerk (the time derivative of acceleration magnitude) are appended to the nine raw channels. Magnitudes are **orientation-invariant**: they do not change when the watch is re-donned at a different rotation, which is the principal nuisance variable for a wrist-worn sensor.

**Augmentation, applied to the training fold only.** Four transforms compose: a random rotation of up to 8° applied jointly to the accelerometer and gyroscope triads, Gaussian jitter (σ = 0.03), magnitude warping (σ = 0.15) and time warping (σ = 0.15).

A caveat stated rather than hidden: the rotation is applied to the accelerometer and gyroscope channels but **not** to Roll/Pitch/Yaw, which are orientation channels in the same frame. Augmented samples therefore carry orientation columns that are not consistent with their rotated inertial columns. The effect is a mild, physically implausible perturbation of the orientation channels — a form of noise rather than a correctness bug — but a stricter implementation would rotate all three triads together.

In [4]:
# ==============================================================================
# PHASE 2: TEMPORAL NORMALIZATION
# ==============================================================================

def interpolate_series(series, target_length=150):
    # METHODOLOGY ALIGNMENT: Resampling to a uniform length of 150 time-steps
    return np.interp(
        np.linspace(0, len(series) - 1, target_length),
        np.arange(len(series)),
        series
    )

# ==============================================================================
# PHASE 2b: FEATURE ENGINEERING
# ==============================================================================

GYRO_IDX = (3, 4, 5)
ACCEL_IDX = (6, 7, 8)

def add_derived_features(raw_rep, gyro_idx=GYRO_IDX, accel_idx=ACCEL_IDX, fs=100):
    """
    WARNING: This changes the array shape from (150, 9) to (150, 12).
    Ensure the CNN-LSTM input layer is updated to match.
    """
    accel = raw_rep[:, list(accel_idx)]
    gyro = raw_rep[:, list(gyro_idx)]

    accel_mag = np.linalg.norm(accel, axis=1, keepdims=True)
    gyro_mag = np.linalg.norm(gyro, axis=1, keepdims=True)

    jerk = np.gradient(accel_mag[:, 0]) * fs
    jerk = jerk.reshape(-1, 1)

    return np.concatenate([raw_rep, accel_mag, gyro_mag, jerk], axis=1)


# ==============================================================================
# PHASE 2c: DATA AUGMENTATION
# ==============================================================================

def jitter(X, sigma=0.03):
    return X + np.random.normal(loc=0.0, scale=sigma, size=X.shape)

def magnitude_warp(X, sigma=0.15, knot=4):
    timesteps, channels = X.shape
    orig_steps = np.arange(timesteps)
    warp_steps = np.linspace(0, timesteps - 1, num=knot + 2)
    random_warps = np.random.normal(loc=1.0, scale=sigma, size=(knot + 2, channels))
    warper = np.array([
        CubicSpline(warp_steps, random_warps[:, dim])(orig_steps)
        for dim in range(channels)
    ]).T
    return X * warper

def time_warp(X, sigma=0.15, knot=4):
    timesteps, channels = X.shape
    orig_steps = np.arange(timesteps)
    warp_steps = np.linspace(0, timesteps - 1, num=knot + 2)
    random_warps = np.random.normal(loc=1.0, scale=sigma, size=(knot + 2,))
    warped_time = CubicSpline(warp_steps, warp_steps * random_warps)(orig_steps)
    warped_time = np.clip(warped_time, 0, timesteps - 1)

    warped = np.zeros_like(X)
    for dim in range(channels):
        warped[:, dim] = np.interp(orig_steps, warped_time, X[:, dim])
    return warped

def random_rotation_matrix(max_angle_deg=8.0):
    angle = np.deg2rad(np.random.uniform(-max_angle_deg, max_angle_deg))
    axis = np.random.normal(size=3)
    axis = axis / np.linalg.norm(axis)
    K = np.array([
        [0, -axis[2], axis[1]],
        [axis[2], 0, -axis[0]],
        [-axis[1], axis[0], 0]
    ])
    R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)
    return R

def augment_raw_rep(raw_rep, gyro_idx=GYRO_IDX, accel_idx=ACCEL_IDX,
                     sigma_jitter=0.03, sigma_mag=0.15, sigma_time=0.15,
                     max_rot_deg=8.0):
    aug = raw_rep.copy()

    R = random_rotation_matrix(max_rot_deg)
    aug[:, list(accel_idx)] = aug[:, list(accel_idx)] @ R.T
    aug[:, list(gyro_idx)] = aug[:, list(gyro_idx)] @ R.T

    aug = jitter(aug, sigma_jitter)
    aug = magnitude_warp(aug, sigma_mag)
    aug = time_warp(aug, sigma_time)
    return aug

## 3. The dual-head architecture

```
Input (150 × 12)
  └─ Conv1D(32, k=3, relu) → MaxPool(2) → Dropout(0.2)
       └─ LSTM(64) → Dropout(0.2)
            ├─ Dense(32, relu) → Dense(n_classes, softmax)   # exercise taxonomy
            └─ Dense(32, relu) → Dense(1, sigmoid)           # execution quality
```

A shared convolutional–recurrent trunk feeds two task-specific heads. The design assumption is that the features distinguishing *which* exercise is being performed and those distinguishing *how well* overlap substantially — so they should be learned jointly rather than by two separate models.

The ablation in section 7 tests that assumption directly, and the answer is partly negative: the shared trunk is not what carries taxonomy performance. Read section 7 before accepting the multi-task framing.

In [5]:

# ==============================================================================
# PHASE 3: DUAL-HEADED DEEP LEARNING ARCHITECTURE
# ==============================================================================

def build_dual_headed_model(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv1D(filters=32, kernel_size=3, activation='relu')(inputs)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.2)(x)

    exercise_head = layers.Dense(32, activation='relu')(x)
    exercise_output = layers.Dense(num_classes, activation='softmax', name='exercise_output')(exercise_head)

    quality_head = layers.Dense(32, activation='relu')(x)
    quality_output = layers.Dense(1, activation='sigmoid', name='quality_output')(quality_head)

    model = models.Model(inputs=inputs, outputs=[exercise_output, quality_output])
    model.compile(
        optimizer='adam',
        loss=['sparse_categorical_crossentropy', 'binary_crossentropy'],
        metrics=['accuracy', 'accuracy']
    )
    return model


## 4. Ablation architectures

Two reduced models isolate the contribution of each half of the trunk:

- **CNN-only** — the LSTM is replaced by `Flatten()`, destroying chronological ordering while preserving spatial structure across channels.
- **LSTM-only** — the convolutional block is removed, so the recurrent layer consumes raw channels with no learned spatial filtering.

Both keep the dual-head output, so they remain comparable to the full model on both tasks. A non-neural **DTW 1-nearest-neighbour** classifier (section 8) provides a floor that involves no learning at all.

In [6]:
from tslearn.neighbors import KNeighborsTimeSeriesClassifier

# ==============================================================================
# PHASE 3B: ABLATION STUDY ARCHITECTURES
# ==============================================================================

def build_cnn_only_model(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    # Spatial extraction only
    x = layers.Conv1D(filters=32, kernel_size=3, activation='relu')(inputs)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Flatten()(x) # Flattening the chronological decay

    ex_head = layers.Dense(32, activation='relu')(x)
    ex_out = layers.Dense(num_classes, activation='softmax', name='ex_out')(ex_head)
    q_head = layers.Dense(32, activation='relu')(x)
    q_out = layers.Dense(1, activation='sigmoid', name='q_out')(q_head)

    model = models.Model(inputs=inputs, outputs=[ex_out, q_out])
    model.compile(
    optimizer='adam',
    loss=['sparse_categorical_crossentropy', 'binary_crossentropy'],
    metrics=[['accuracy'], ['accuracy']]
)
    return model

def build_lstm_only_model(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    # Temporal sequence modeling only
    x = layers.LSTM(64, return_sequences=False)(inputs)
    x = layers.Dropout(0.2)(x)

    ex_head = layers.Dense(32, activation='relu')(x)
    ex_out = layers.Dense(num_classes, activation='softmax', name='ex_out')(ex_head)
    q_head = layers.Dense(32, activation='relu')(x)
    q_out = layers.Dense(1, activation='sigmoid', name='q_out')(q_head)

    model = models.Model(inputs=inputs, outputs=[ex_out, q_out])
    model.compile(
    optimizer='adam',
    loss=['sparse_categorical_crossentropy', 'binary_crossentropy'],
    metrics=[['accuracy'], ['accuracy']]
)
    return model

## 5. Pipeline execution and leakage-free LOSO validation

This is the cell that produces the headline numbers, and the ordering of operations inside it is the methodological core of the paper.

**Ground-truth generation.** There are no therapist-annotated quality labels. For each exercise class, a multivariate DTW barycentre is computed over the *training* repetitions of that class; each repetition's DTW distance to that centroid is measured; and repetitions beyond **mean + 1.5σ** are labelled *degraded*, the rest *optimal*.

Two consequences follow, and both are stated in the paper:

- These are **pseudo-labels** — an unsupervised proxy for kinematic deviation, not clinical ground truth. The model is being trained to reproduce a DTW threshold rule, and cannot outperform the rule that generated its targets. Validating these labels against EMG or therapist-annotated video is named as future work.
- The threshold is computed on **training** distances and applied unchanged to the held-out subject. That is the correct direction — but it means a participant whose form is idiosyncratic yet consistent will be flagged against a centroid built from *other people*.

**Execution order, which is what makes the fold leakage-free:**

```
split raw → augment training subset only → compute derived features → fit scaler on training only
```

Every step that could carry information across the boundary happens after the boundary is drawn. The scaler is fitted on the augmented training subset and merely applied to the validation and test partitions.

**Training.** Class weights are computed separately for both heads and passed as sample weights, since both the exercise distribution and the optimal/degraded split are imbalanced. Early stopping (patience 5, best weights restored) and `ReduceLROnPlateau` (factor 0.5, patience 3) run on the validation partition of the training fold.

In [7]:
# ==============================================================================
# PHASE 4: PIPELINE EXECUTION & CLINICAL VALIDATION (CORRECTED)
# ==============================================================================

# 1. INJECT MISSING AUGMENTATION HELPER FUNCTION
def augment_training_fold(X_raw, y_ex, y_q, sw, n_augments=2):
    X_aug, y_ex_aug, y_q_aug, sw_aug = [], [], [], []
    for i in range(len(X_raw)):
        # Original repetition
        X_aug.append(X_raw[i])
        y_ex_aug.append(y_ex[i])
        y_q_aug.append(y_q[i])
        sw_aug.append(sw[i])

        # Synthetic augmented clones
        for _ in range(n_augments):
            X_aug.append(augment_raw_rep(X_raw[i]))
            y_ex_aug.append(y_ex[i])
            y_q_aug.append(y_q[i])
            sw_aug.append(sw[i])

    return np.array(X_aug), np.array(y_ex_aug), np.array(y_q_aug), np.array(sw_aug)

if __name__ == "__main__":
    print("Initializing pipeline (Leakage-Free Validation)...")

    feature_columns = ['Roll', 'Pitch', 'Yaw', 'RotationX', 'RotationY', 'RotationZ',
                       'AccelerationX', 'AccelerationY', 'AccelerationZ']

    N_AUGMENTS_PER_REP = 2

    DATA_DIR = "data"                      # place the CSV here (not included in this repository)
    file_path = pd.read_csv(f"{DATA_DIR}/Rehab_Data_Full_Rebuilt.csv")
    print("Loading data...")
    df_raw = file_path

    df_raw = df_raw.dropna(subset=['Subject_ID', 'Exercise'])

    # NO SUBJECTS EXCLUDED: All subjects present in the CSV will be processed
    print(f"Active subjects loaded: {df_raw['Subject_ID'].unique()}")

    print("File loaded successfully. Beginning segmentation and DTW ground truth generation...")

    all_processed_reps = []
    all_exercise_labels = []
    all_subject_groups = []

    for (subject, exercise), group in df_raw.groupby(['Subject_ID', 'Exercise']):
        reps = segment_repetitions(group, feature_cols=feature_columns, fs=100)
        # Note: The len(reps) > 2 filter was removed per previous correction to prevent survivorship bias
        for rep in reps:
            resampled_feat = np.stack([interpolate_series(rep[col].values, 150) for col in feature_columns], axis=-1)
            all_processed_reps.append(resampled_feat)
            all_exercise_labels.append(exercise)
            all_subject_groups.append(subject)

    X_raw = np.array(all_processed_reps)
    le = LabelEncoder()
    y_exercise = le.fit_transform(all_exercise_labels)
    subjects = np.array(all_subject_groups)
    print(f"Data Prepared. Total repetition samples generated: {X_raw.shape[0]}")

    X_full = np.array([add_derived_features(rep) for rep in X_raw])

    logo = LeaveOneGroupOut()
    tracker_ex_acc, tracker_ex_f1_macro = [], []
    tracker_q_acc, tracker_q_auc = [], []

    agg_y_true_tax, agg_y_pred_tax = [], []
    agg_y_true_qual, agg_y_pred_qual = [], []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X_raw, y_exercise, groups=subjects)):
        test_subject = subjects[test_idx][0]
        print(f"\n[{fold + 1}/{len(np.unique(subjects))}] Training | Holding out Subject: {test_subject}...")

        tf.keras.backend.clear_session()

        X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
        X_train_full, X_test_full = X_full[train_idx], X_full[test_idx]
        y_ex_train, y_ex_test = y_exercise[train_idx], y_exercise[test_idx]

        y_q_train = np.zeros(len(y_ex_train))
        y_q_test = np.zeros(len(y_ex_test))

        print("   -> Generating dynamic DTW baselines per exercise...")
        for ex_class in np.unique(y_ex_train):
            ex_train_mask = (y_ex_train == ex_class)
            X_train_ex = X_train_raw[ex_train_mask]
            if len(X_train_ex) == 0:
                continue

            centroid = dtw_barycenter_averaging(X_train_ex, max_iter=5)
            train_dists = np.array([dtw(rep, centroid) for rep in X_train_ex])
            threshold = np.mean(train_dists) + (1.5 * np.std(train_dists))
            y_q_train[ex_train_mask] = np.where(train_dists > threshold, 1, 0)
            ex_test_mask = (y_ex_test == ex_class)
            X_test_ex = X_test_raw[ex_test_mask]
            if len(X_test_ex) > 0:
                test_dists = np.array([dtw(rep, centroid) for rep in X_test_ex])
                y_q_test[ex_test_mask] = np.where(test_dists > threshold, 1, 0)

        quality_classes = np.unique(y_q_train)
        q_weights = compute_class_weight(class_weight='balanced', classes=quality_classes, y=y_q_train)
        sw_quality = np.ones(len(y_q_train))
        for cls, weight in zip(quality_classes, q_weights):
            sw_quality[y_q_train == cls] = weight

        tax_classes = np.unique(y_ex_train)
        t_weights = compute_class_weight(class_weight='balanced', classes=tax_classes, y=y_ex_train)
        sw_taxonomy = np.ones(len(y_ex_train))
        for cls, weight in zip(tax_classes, t_weights):
            sw_taxonomy[y_ex_train == cls] = weight

        # --- CORRECTED ORDER: SPLIT -> AUGMENT -> FEATURES -> SCALE ---

        # 1. SPLIT RAW DATA FIRST (Isolate validation completely)
        X_t_raw, X_v_raw, y_ex_t, y_ex_v, y_q_t, y_q_v, sw_t_t, sw_t_v, sw_q_t, sw_q_v = train_test_split(
            X_train_raw, y_ex_train, y_q_train, sw_taxonomy, sw_quality,
            test_size=0.2, random_state=42
        )

        # 2. AUGMENT ONLY THE TRAINING SUBSET
        print(f"   -> Augmenting isolated training fold ({N_AUGMENTS_PER_REP}x per rep)...")
        X_t_raw_aug, y_ex_t_aug, y_q_t_aug, sw_q_t_aug = augment_training_fold(
            X_t_raw, y_ex_t, y_q_t, sw_q_t, n_augments=N_AUGMENTS_PER_REP
        )
        sw_t_t_aug = np.repeat(sw_t_t, N_AUGMENTS_PER_REP + 1)
        print(f"   -> Train set expanded: {len(X_t_raw)} -> {len(X_t_raw_aug)} reps")

        # 3. COMPUTE DERIVED FEATURES INDEPENDENTLY
        X_t_full_aug = np.array([add_derived_features(rep) for rep in X_t_raw_aug])
        X_v_full = np.array([add_derived_features(rep) for rep in X_v_raw])
        # Note: X_test_full is already pre-computed above from X_full[test_idx]

        # 4. SCALE USING ONLY TRAINING STATISTICS
        scaler = StandardScaler()
        num_train, timesteps, features = X_t_full_aug.shape
        num_val = X_v_full.shape[0]
        num_test = X_test_full.shape[0]

        X_t = scaler.fit_transform(X_t_full_aug.reshape(-1, features)).reshape(num_train, timesteps, features)
        X_v = scaler.transform(X_v_full.reshape(-1, features)).reshape(num_val, timesteps, features)
        X_test_scaled = scaler.transform(X_test_full.reshape(-1, features)).reshape(num_test, timesteps, features)

        # --- TRAINING ---
        model = build_dual_headed_model(input_shape=(timesteps, features), num_classes=len(le.classes_))

        callbacks = [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0),
                     ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=0)]

        model.fit(
            x=X_t,
            y=[y_ex_t_aug, y_q_t_aug],
            sample_weight=[sw_t_t_aug, sw_q_t_aug],
            validation_data=(X_v, [y_ex_v, y_q_v], [sw_t_v, sw_q_v]),
            epochs=30,
            batch_size=32,
            callbacks=callbacks,
            verbose=0
        )

        predictions = model.predict(X_test_scaled, verbose=0)
        y_ex_pred = np.argmax(predictions[0], axis=1)
        y_q_pred = np.round(predictions[1].flatten()).astype(int)

        agg_y_true_tax.extend(y_ex_test)
        agg_y_pred_tax.extend(y_ex_pred)
        agg_y_true_qual.extend(y_q_test)
        agg_y_pred_qual.extend(y_q_pred)

        ex_acc = accuracy_score(y_ex_test, y_ex_pred)
        ex_f1 = f1_score(y_ex_test, y_ex_pred, average='macro', zero_division=0)
        q_acc = accuracy_score(y_q_test, y_q_pred)
        try:
            q_auc = roc_auc_score(y_q_test, predictions[1].flatten())
        except ValueError:
            q_auc = np.nan

        tracker_ex_acc.append(ex_acc)
        tracker_ex_f1_macro.append(ex_f1)
        tracker_q_acc.append(q_acc)
        if not np.isnan(q_auc):
            tracker_q_auc.append(q_auc)

        print(f"   -> Fold Complete. Tax Acc: {ex_acc:.2f} | Qual Acc: {q_acc:.2f}")

    print("\n" + "=" * 70)
    print(" FINAL LOSO CROSS-VALIDATION RESULTS (ALL SUBJECTS) ")
    print("=" * 70)
    print(f"Exercise Classification Accuracy:  {np.mean(tracker_ex_acc):.4f} ± {np.std(tracker_ex_acc):.4f}")
    print(f"Exercise Classification Macro-F1:  {np.mean(tracker_ex_f1_macro):.4f} ± {np.std(tracker_ex_f1_macro):.4f}")
    print("-" * 70)
    print(f"Execution Quality Accuracy:        {np.mean(tracker_q_acc):.4f} ± {np.std(tracker_q_acc):.4f}")
    print(f"Execution Quality ROC-AUC:         {np.mean(tracker_q_auc):.4f} ± {np.std(tracker_q_auc):.4f}")
    print("=" * 70)

Initializing pipeline (Leakage-Free Validation)...
Loading data...
Active subjects loaded: <StringArray>
['Subject_01', 'Subject_02', 'Subject_03', 'Subject_04', 'Subject_05',
 'Subject_06', 'Subject_07', 'Subject_08', 'Subject_09', 'Subject_10',
 'Subject_11', 'Subject_12', 'Subject_13', 'Subject_14', 'Subject_15']
Length: 15, dtype: str
File loaded successfully. Beginning segmentation and DTW ground truth generation...
Data Prepared. Total repetition samples generated: 3001

[1/15] Training | Holding out Subject: Subject_01...

   -> Generating dynamic DTW baselines per exercise...
   -> Augmenting isolated training fold (2x per rep)...
   -> Train set expanded: 2258 -> 6774 reps
   -> Fold Complete. Tax Acc: 0.92 | Qual Acc: 0.93

[2/15] Training | Holding out Subject: Subject_02...
   -> Generating dynamic DTW baselines per exercise...
   -> Augmenting isolated training fold (2x per rep)...
   -> Train set expanded: 2155 -> 6465 reps
   -> Fold Complete. Tax Acc: 0.74 | Qual Acc: 0

### Results — strict LOSO across 15 subjects

| Metric | Value |
|---|---|
| Exercise classification accuracy | **88.73% ± 6.00%** |
| Exercise classification macro-F1 | 82.87% ± 13.78% |
| Execution quality accuracy | **88.99% ± 4.66%** |
| Execution quality ROC-AUC | **0.9060 ± 0.0553** |

The macro-F1 spread (±13.78%) is much wider than the accuracy spread (±6.00%). That gap is the signature of uneven per-class performance: accuracy is carried by the well-represented exercises while the minority classes vary considerably from fold to fold. The paper separately reports a precision of **0.542** on the *degraded* class — meaning roughly one in two degradation alerts would be a false positive, which is the single most important number for anyone considering deployment.

## 6. Confusion matrices

Aggregated across all 15 folds — every prediction shown was made on a participant held out of training. Figures are written to `figures/` at 300 dpi.

In [8]:
# ==============================================================================
# PHASE 5: HIGH-RESOLUTION PUBLICATION FIGURE GENERATION (LOCAL / VS CODE)
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np
import os

# Local output directory (created if it doesn't exist)
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def generate_high_res_matrices(y_true_tax, y_pred_tax, y_true_qual, y_pred_qual, class_labels):
    """
    class_labels: pass `le.classes_` from Phase 4's LabelEncoder so the tick
    labels are guaranteed to match the integer-encoded classes actually
    produced by confusion_matrix (sorted ascending == alphabetical order
    from LabelEncoder). Do not hand-type/reorder this list separately.
    """
    plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold'})

    # 1. Taxonomy Classification Matrix
    plt.figure(figsize=(16, 14))
    cm_tax = confusion_matrix(y_true_tax, y_pred_tax)

    if cm_tax.shape[0] != len(class_labels):
        raise ValueError(
            f"confusion_matrix has {cm_tax.shape[0]} classes but "
            f"{len(class_labels)} labels were provided. Some class may be "
            f"missing from the aggregated true/pred arrays (e.g. an exercise "
            f"with zero test reps across all folds)."
        )

    ax1 = sns.heatmap(cm_tax, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels,
                yticklabels=class_labels,
                linewidths=1.5, linecolor='black',
                annot_kws={"size": 16, "weight": "bold"},
                cbar_kws={'shrink': .8})

    ax1.collections[0].colorbar.outline.set_linewidth(1.5)

    plt.title('Aggregate Taxonomy Classification Confusion Matrix', fontsize=22, fontweight='bold', pad=25)
    plt.ylabel('True Exercise Class', fontsize=18, fontweight='bold')
    plt.xlabel('Predicted Exercise Class', fontsize=18, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=14, fontweight='bold')
    plt.yticks(rotation=0, fontsize=14, fontweight='bold')
    plt.tight_layout()

    tax_path = os.path.join(OUTPUT_DIR, 'Taxonomy_Confusion_Matrix_300dpi.png')
    plt.savefig(tax_path, dpi=300, bbox_inches='tight')
    plt.close()

    # 2. Execution Quality Assessment Matrix
    plt.figure(figsize=(10, 8))
    cm_qual = confusion_matrix(y_true_qual, y_pred_qual)

    ax2 = sns.heatmap(cm_qual, annot=True, fmt='d', cmap='YlGnBu',
                xticklabels=['Optimal (0)', 'Degraded (1)'],
                yticklabels=['Optimal (0)', 'Degraded (1)'],
                linewidths=2.0, linecolor='black',
                annot_kws={"size": 24, "weight": "bold"})

    ax2.collections[0].colorbar.outline.set_linewidth(1.5)

    plt.title('Aggregate Execution Quality Confusion Matrix', fontsize=20, fontweight='bold', pad=25)
    plt.ylabel('True Execution State', fontsize=18, fontweight='bold')
    plt.xlabel('Predicted Execution State', fontsize=18, fontweight='bold')
    plt.xticks(fontsize=16, fontweight='bold')
    plt.yticks(fontsize=16, rotation=0, fontweight='bold')
    plt.tight_layout()

    qual_path = os.path.join(OUTPUT_DIR, 'Quality_Confusion_Matrix_300dpi.png')
    plt.savefig(qual_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"High-resolution matrices saved to:")
    print(f"  {tax_path}")
    print(f"  {qual_path}")


# Execute plotting using aggregated arrays from Phase 4
# `le` is the LabelEncoder fit in Phase 4 — reuse it directly rather than
# retyping TARGET_EXERCISES, to guarantee label order matches the matrix.
generate_high_res_matrices(
    np.array(agg_y_true_tax),
    np.array(agg_y_pred_tax),
    np.array(agg_y_true_qual),
    np.array(agg_y_pred_qual),
    class_labels=le.classes_
)

High-resolution matrices saved to:
  figures/Taxonomy_Confusion_Matrix_300dpi.png
  figures/Quality_Confusion_Matrix_300dpi.png


## 7. Neural ablation — and what it actually shows

| Architecture | Taxonomy acc. | Taxonomy F1 | Quality ROC-AUC |
|---|---|---|---|
| LSTM-only | 60.06% | 0.5418 | 0.7175 |
| CNN-only | **88.96%** | **0.8445** | 0.8639 |
| Proposed CNN–LSTM | 88.73% | 0.8287 | **0.9060** |

Read the taxonomy column honestly: **the CNN-only baseline matches and very slightly exceeds the full hybrid (88.96% vs 88.73%), with a higher macro-F1.** For deciding which exercise is being performed, the LSTM contributes nothing measurable. Spatial feature extraction alone is sufficient.

The quality column is where the hybrid earns its place: 0.9060 against 0.8639, a gain of roughly 0.042 AUC. Detecting *how well* a movement is executed requires modelling how it evolves over time in a way that detecting *which* movement it is does not.

The LSTM-only collapse to 60.06% is equally informative in the other direction — a recurrent layer fed raw channels without convolutional filtering cannot separate the exercise classes at all.

So the defensible claim is narrow and specific: **the hybrid architecture is justified by the execution-quality task, not by the taxonomy task.** Any summary that reports only the 88.73% headline and the architecture diagram is overstating what the ablation supports.

In [9]:
# ==============================================================================
# NEURAL ABLATION STUDY (CNN-ONLY & LSTM-ONLY) — CORRECTED
# ==============================================================================
print("\n" + "=" * 70)
print(" EXECUTING NEURAL ABLATION STUDY (LEAKAGE-FREE) ")
print("=" * 70)

# Trackers for CNN-Only Baseline
cnn_ex_acc, cnn_ex_f1, cnn_q_auc = [], [], []
# Trackers for LSTM-Only Baseline
lstm_ex_acc, lstm_ex_f1, lstm_q_auc = [], [], []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_raw, y_exercise, groups=subjects)):
    test_subject = subjects[test_idx][0]
    print(f"\n[{fold + 1}/{len(np.unique(subjects))}] Ablation Fold | Holding out: {test_subject}...")

    # --- 1. DATA PREP (Calculated ONCE per fold) ---
    X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
    X_train_full, X_test_full = X_full[train_idx], X_full[test_idx]
    y_ex_train, y_ex_test = y_exercise[train_idx], y_exercise[test_idx]

    y_q_train = np.zeros(len(y_ex_train))
    y_q_test = np.zeros(len(y_ex_test))

    for ex_class in np.unique(y_ex_train):
        ex_train_mask = (y_ex_train == ex_class)
        X_train_ex = X_train_raw[ex_train_mask]
        if len(X_train_ex) == 0:
            continue

        centroid = dtw_barycenter_averaging(X_train_ex, max_iter=5)
        train_dists = np.array([dtw(rep, centroid) for rep in X_train_ex])
        threshold = np.mean(train_dists) + (1.5 * np.std(train_dists))
        y_q_train[ex_train_mask] = np.where(train_dists > threshold, 1, 0)

        ex_test_mask = (y_ex_test == ex_class)
        X_test_ex = X_test_raw[ex_test_mask]
        if len(X_test_ex) > 0:
            test_dists = np.array([dtw(rep, centroid) for rep in X_test_ex])
            y_q_test[ex_test_mask] = np.where(test_dists > threshold, 1, 0)

    # --- Class weights: map by actual class VALUE, not position, since a
    # LOSO fold's y_ex_train can be missing one or more exercise classes ---
    quality_classes = np.unique(y_q_train)
    q_weights = compute_class_weight('balanced', classes=quality_classes, y=y_q_train)
    sw_quality = np.ones(len(y_q_train))
    for cls, weight in zip(quality_classes, q_weights):
        sw_quality[y_q_train == cls] = weight

    tax_classes = np.unique(y_ex_train)
    t_weights = compute_class_weight('balanced', classes=tax_classes, y=y_ex_train)
    sw_taxonomy = np.ones(len(y_ex_train))
    for cls, weight in zip(tax_classes, t_weights):
        sw_taxonomy[y_ex_train == cls] = weight

    # --- CORRECTED ORDER: SPLIT -> AUGMENT -> FEATURES -> SCALE ---

    # 1. SPLIT RAW DATA FIRST (Isolate validation completely)
    X_t_raw, X_v_raw, y_ex_t, y_ex_v, y_q_t, y_q_v, sw_t_t, sw_t_v, sw_q_t, sw_q_v = train_test_split(
        X_train_raw, y_ex_train, y_q_train, sw_taxonomy, sw_quality,
        test_size=0.2, random_state=42
    )

    # 2. AUGMENT ONLY THE TRAINING SUBSET
    X_t_raw_aug, y_ex_t_aug, y_q_t_aug, sw_q_t_aug = augment_training_fold(
        X_t_raw, y_ex_t, y_q_t, sw_q_t, n_augments=N_AUGMENTS_PER_REP
    )
    sw_t_t_aug = np.repeat(sw_t_t, N_AUGMENTS_PER_REP + 1)

    # 3. COMPUTE DERIVED FEATURES INDEPENDENTLY
    X_t_full_aug = np.array([add_derived_features(rep) for rep in X_t_raw_aug])
    X_v_full = np.array([add_derived_features(rep) for rep in X_v_raw])
    # X_test_full is already pre-computed above from X_full[test_idx]

    # 4. SCALE USING ONLY TRAINING STATISTICS
    scaler = StandardScaler()
    num_train, timesteps, features = X_t_full_aug.shape
    num_val = X_v_full.shape[0]
    num_test = X_test_full.shape[0]

    X_t = scaler.fit_transform(X_t_full_aug.reshape(-1, features)).reshape(num_train, timesteps, features)
    X_v = scaler.transform(X_v_full.reshape(-1, features)).reshape(num_val, timesteps, features)
    X_test_scaled = scaler.transform(X_test_full.reshape(-1, features)).reshape(num_test, timesteps, features)

    callbacks = [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0),
                 ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=0)]

    # --- 2. TRAIN CNN-ONLY BASELINE ---
    tf.keras.backend.clear_session()
    print("   -> Training CNN-Only Architecture...")
    cnn_model = build_cnn_only_model(input_shape=(timesteps, features), num_classes=len(le.classes_))
    cnn_model.fit(x=X_t, y=[y_ex_t_aug, y_q_t_aug], sample_weight=[sw_t_t_aug, sw_q_t_aug],
                  validation_data=(X_v, [y_ex_v, y_q_v], [sw_t_v, sw_q_v]),
                  epochs=30, batch_size=32, callbacks=callbacks, verbose=0)

    preds_cnn = cnn_model.predict(X_test_scaled, verbose=0)
    y_ex_pred_cnn = np.argmax(preds_cnn[0], axis=1)
    cnn_ex_acc.append(accuracy_score(y_ex_test, y_ex_pred_cnn))
    cnn_ex_f1.append(f1_score(y_ex_test, y_ex_pred_cnn, average='macro', zero_division=0))
    try:
        cnn_q_auc.append(roc_auc_score(y_q_test, preds_cnn[1].flatten()))
    except ValueError:
        pass

    # --- 3. TRAIN LSTM-ONLY BASELINE ---
    tf.keras.backend.clear_session()
    print("   -> Training LSTM-Only Architecture...")
    lstm_model = build_lstm_only_model(input_shape=(timesteps, features), num_classes=len(le.classes_))
    lstm_model.fit(x=X_t, y=[y_ex_t_aug, y_q_t_aug], sample_weight=[sw_t_t_aug, sw_q_t_aug],
                   validation_data=(X_v, [y_ex_v, y_q_v], [sw_t_v, sw_q_v]),
                   epochs=30, batch_size=32, callbacks=callbacks, verbose=0)

    preds_lstm = lstm_model.predict(X_test_scaled, verbose=0)
    y_ex_pred_lstm = np.argmax(preds_lstm[0], axis=1)
    lstm_ex_acc.append(accuracy_score(y_ex_test, y_ex_pred_lstm))
    lstm_ex_f1.append(f1_score(y_ex_test, y_ex_pred_lstm, average='macro', zero_division=0))
    try:
        lstm_q_auc.append(roc_auc_score(y_q_test, preds_lstm[1].flatten()))
    except ValueError:
        pass

    print(f"   -> Fold Complete. CNN Acc: {cnn_ex_acc[-1]:.2f} | LSTM Acc: {lstm_ex_acc[-1]:.2f}")

print("\n" + "=" * 70)
print(" FINAL ABLATION RESULTS (CNN-ONLY vs LSTM-ONLY) ")
print("=" * 70)
print(f"[CNN-Only] Taxonomy Acc:  {np.mean(cnn_ex_acc):.4f} \u00b1 {np.std(cnn_ex_acc):.4f}")
print(f"[CNN-Only] Taxonomy F1:   {np.mean(cnn_ex_f1):.4f} \u00b1 {np.std(cnn_ex_f1):.4f}")
print(f"[CNN-Only] Quality AUC:   {np.mean(cnn_q_auc):.4f} \u00b1 {np.std(cnn_q_auc):.4f}  (n={len(cnn_q_auc)}/{len(cnn_ex_acc)} folds)")
print("-" * 70)
print(f"[LSTM-Only] Taxonomy Acc: {np.mean(lstm_ex_acc):.4f} \u00b1 {np.std(lstm_ex_acc):.4f}")
print(f"[LSTM-Only] Taxonomy F1:  {np.mean(lstm_ex_f1):.4f} \u00b1 {np.std(lstm_ex_f1):.4f}")
print(f"[LSTM-Only] Quality AUC:  {np.mean(lstm_q_auc):.4f} \u00b1 {np.std(lstm_q_auc):.4f}  (n={len(lstm_q_auc)}/{len(lstm_ex_acc)} folds)")
print("=" * 70)


 EXECUTING NEURAL ABLATION STUDY (LEAKAGE-FREE) 

[1/15] Ablation Fold | Holding out: Subject_01...
   -> Training CNN-Only Architecture...
   -> Training LSTM-Only Architecture...
   -> Fold Complete. CNN Acc: 0.83 | LSTM Acc: 0.60

[2/15] Ablation Fold | Holding out: Subject_02...
   -> Training CNN-Only Architecture...
   -> Training LSTM-Only Architecture...
   -> Fold Complete. CNN Acc: 0.82 | LSTM Acc: 0.50

[3/15] Ablation Fold | Holding out: Subject_03...
   -> Training CNN-Only Architecture...
   -> Training LSTM-Only Architecture...
   -> Fold Complete. CNN Acc: 0.95 | LSTM Acc: 0.75

[4/15] Ablation Fold | Holding out: Subject_04...
   -> Training CNN-Only Architecture...
   -> Training LSTM-Only Architecture...
   -> Fold Complete. CNN Acc: 0.98 | LSTM Acc: 0.79

[5/15] Ablation Fold | Holding out: Subject_05...
   -> Training CNN-Only Architecture...
   -> Training LSTM-Only Architecture...
   -> Fold Complete. CNN Acc: 0.96 | LSTM Acc: 0.67

[6/15] Ablation Fold | Holdin

## 8. Standalone DTW baseline (1-nearest neighbour)

| | Taxonomy acc. | Taxonomy macro-F1 |
|---|---|---|
| DTW 1-NN | **88.56% ± 5.60%** | 0.8344 ± 0.1313 |
| Proposed CNN–LSTM | 88.73% ± 6.00% | 0.8287 ± 0.1378 |

A 1-nearest-neighbour classifier over raw DTW distances — no learning, no augmentation, no scaling — reaches 88.56% on exercise taxonomy against the deep model's 88.73%, and a *higher* macro-F1.

This is the sharpest version of section 7's finding. On taxonomy alone, the deep model is not beating a non-parametric baseline. What it offers instead is the second head: DTW 1-NN produces no execution-quality assessment at all, and cannot without a separate mechanism. It is also expensive at inference time, since every query must be compared against the full training set — the opposite of what a wrist-worn device needs.

The case for the neural model is multi-task capability and bounded inference cost, not raw taxonomy accuracy. That is a real case, but it is a different one from the one a taxonomy-accuracy table alone would suggest.

In [10]:
from tslearn.neighbors import KNeighborsTimeSeriesClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut
import numpy as np

# ==============================================================================
# ABLATION STUDY: STANDALONE DTW (1-NEAREST NEIGHBOR)
# ==============================================================================

print("\n" + "=" * 70)
print(" EXECUTING STANDALONE DTW (1-NN) ABLATION ")
print("=" * 70)

dtw_acc_tracker, dtw_f1_tracker = [], []

# Initialize the LOSO splitter here so the cell can run independently
logo = LeaveOneGroupOut()

# We use the same Leave-One-Subject-Out splits to ensure fair comparison
for fold, (train_idx, test_idx) in enumerate(logo.split(X_raw, y_exercise, groups=subjects)):
    test_subject = subjects[test_idx][0]
    print(f"\n[{fold + 1}/{len(np.unique(subjects))}] DTW-1NN | Testing Subject: {test_subject}...")

    # DTW 1-NN relies purely on raw spatial inputs (no augmentation or scaling)
    X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
    y_ex_train, y_ex_test = y_exercise[train_idx], y_exercise[test_idx]

    # Initialize the DTW Classifier (n_jobs=-1 uses all CPU cores to speed this up)
    knn_dtw = KNeighborsTimeSeriesClassifier(n_neighbors=1, metric="dtw", n_jobs=-1)
    knn_dtw.fit(X_train_raw, y_ex_train)

    y_pred = knn_dtw.predict(X_test_raw)

    fold_acc = accuracy_score(y_ex_test, y_pred)
    fold_f1 = f1_score(y_ex_test, y_pred, average='macro', zero_division=0)

    dtw_acc_tracker.append(fold_acc)
    dtw_f1_tracker.append(fold_f1)
    print(f"   -> Tax Acc: {fold_acc:.4f} | Tax F1: {fold_f1:.4f}")

print("\n" + "=" * 50)
print(" FINAL DTW-1NN LOSO-CV RESULTS (ALL 15 SUBJECTS) ")
print("=" * 50)
print(f"Taxonomy Accuracy: {np.mean(dtw_acc_tracker):.4f} ± {np.std(dtw_acc_tracker):.4f}")
print(f"Taxonomy Macro-F1: {np.mean(dtw_f1_tracker):.4f} ± {np.std(dtw_f1_tracker):.4f}")
print("=" * 50)


 EXECUTING STANDALONE DTW (1-NN) ABLATION 

[1/15] DTW-1NN | Testing Subject: Subject_01...
   -> Tax Acc: 0.9213 | Tax F1: 0.8725

[2/15] DTW-1NN | Testing Subject: Subject_02...
   -> Tax Acc: 0.7427 | Tax F1: 0.7615

[3/15] DTW-1NN | Testing Subject: Subject_03...
   -> Tax Acc: 0.9308 | Tax F1: 0.9432

[4/15] DTW-1NN | Testing Subject: Subject_04...
   -> Tax Acc: 0.9384 | Tax F1: 0.9323

[5/15] DTW-1NN | Testing Subject: Subject_05...
   -> Tax Acc: 0.8140 | Tax F1: 0.7812

[6/15] DTW-1NN | Testing Subject: Subject_06...
   -> Tax Acc: 0.9565 | Tax F1: 0.9444

[7/15] DTW-1NN | Testing Subject: Subject_07...
   -> Tax Acc: 0.8362 | Tax F1: 0.7767

[8/15] DTW-1NN | Testing Subject: Subject_08...
   -> Tax Acc: 0.8931 | Tax F1: 0.8921

[9/15] DTW-1NN | Testing Subject: Subject_09...
   -> Tax Acc: 0.8945 | Tax F1: 0.8936

[10/15] DTW-1NN | Testing Subject: Subject_10...
   -> Tax Acc: 0.8279 | Tax F1: 0.6568

[11/15] DTW-1NN | Testing Subject: Subject_11...
   -> Tax Acc: 0.9195 | T